# Video exogenous controls and population events

Synchronized face-video variables are added to regional increment equations. Population events are then aligned within detrended Awake and NREM episodes to test for reproducible regional recruitment lags.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig
ROOT = Path('..').resolve()
OUT = ROOT / 'figures/261027视频外生输入与群体事件'
OUT.mkdir(parents=True, exist_ok=True)
video = pd.read_csv(ROOT / 'logs/scripts/regional_video_exogenous/initial/model_summary.csv')
events = np.load(ROOT / 'logs/scripts/regional_event_propagation/initial/event_templates.npz', allow_pickle=True)
video

,state,model,alpha,validation_mse,dx_r2_flat,dx_r2_region_median,dx_corr_flat,n_test_targets
0,Awake,zero,0.0,0.121409,-0.000003,-0.000004,NaN,5099
1,Awake,state,100.0,0.109400,0.079824,0.070139,0.299375,5099
2,Awake,video,1000.0,0.120836,0.002498,0.000311,0.067052,5099
3,Awake,state_video,1000.0,0.108224,0.089786,0.084049,0.304254,5099
4,Awake,state_video_history,1000.0,0.107458,0.099784,0.086877,0.320757,5099
5,NREM,zero,0.0,0.207015,-0.000361,-0.000378,NaN,832
6,NREM,state,100.0,0.173350,0.175583,0.129172,0.419134,832
7,NREM,video,1000.0,0.206892,-0.001216,-0.001188,0.001185,832
8,NREM,state_video,100.0,0.173268,0.175564,0.129737,0.419060,832
9,NREM,state_video_history,100.0,0.173447,0.172555,0.132551,0.415625,832


In [2]:
models = ['zero', 'state', 'video', 'state_video', 'state_video_history']
labels = ['zero dx', 'state', 'video', 'state + video', 'state + 1 s video']
colors = {'Awake': '#cc5544', 'NREM': '#4477aa', 'REM': '#aa55aa'}
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
x = np.arange(len(models))
for offset, state in zip([-.22, 0, .22], ['Awake', 'NREM', 'REM']):
    table = video[video.state == state].set_index('model')
    ax.bar(x + offset, table.loc[models].dx_r2_flat, width=.21, color=colors[state], label=state)
ax.axhline(0, color='0.25', lw=.8)
ax.set_xticks(x, labels, rotation=45, ha='right')
ax.set(ylabel=r'Held-out $\Delta x$ $R^2$', title='Exogenous video value')
ax.legend(frameon=False)
ax = next(ax_loader)
template = events['template'].T
offsets = events['offsets_seconds']
regions = events['region_names'].astype(str)
order = np.argsort(template[:, len(offsets)//2])[::-1]
limit = np.quantile(np.abs(template), .98)
ax.imshow(template[order], aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit, extent=[offsets[0], offsets[-1], len(regions)-.5, -.5])
ax.axvline(0, color='black', lw=.8)
ax.set(xlabel='Time from event peak (s)', ylabel='Region (sorted)', title='NREM event-triggered activity')
fig.savefig(OUT / 'video_and_population_events.png')
fig.savefig(OUT / 'video_and_population_events.svg')
fig.savefig(OUT / 'video_and_population_events.pdf')
plt.show()

About one second of video history raises Awake regional increment R² from 0.080 to 0.100. Video alone is weak (0.0025), indicating a modest conditional contribution rather than a replacement for neural state. NREM receives no video benefit, so its strong regional coupling result is not explained by these measured face variables.

NREM event-triggered regional means peak in the same 0.241 s frame. Across 151 events, every regional bootstrap lag interval contains zero. The data support synchronous population events at this resolution, not a reproducible region-level propagation order.